# Lab 7A: Build and test an MCP server
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 7 · about 30 minutes**

**Objectives**
- Write an MCP server with tools and resources
- Test it with the MCP Python client before any model sees it
- Check structured errors and empty results
- Improve a tool description and see the effect

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
!pip -q install -U mcp
import os, json

## 1. The server
Read every docstring: it becomes the tool description. The server supports MCP Python SDK 2.x (`MCPServer`) and 1.x (`FastMCP`).

In [ ]:
%%writefile /content/support_backend.py
"""support-backend MCP server (synthetic data). Works with MCP Python SDK 2.x (MCPServer) and 1.x (FastMCP)."""
import json, os
try:
    from mcp.server.mcpserver import MCPServer as Server
    from mcp.server.mcpserver.exceptions import ToolError
except ImportError:
    from mcp.server.fastmcp import FastMCP as Server
    from mcp.server.fastmcp.exceptions import ToolError

mcp = Server("support-backend")
CUSTOMERS = {"C-1001": {"customer_id": "C-1001", "name": "Asha Menon", "email": "asha.m@example.com", "plan": "Fiber 300"},
             "C-1002": {"customer_id": "C-1002", "name": "Rahul Nair", "email": "rahul.n@example.com", "plan": "Mobile Plus"}}
ORDERS = {"ORD-5001": {"order_id": "ORD-5001", "customer_id": "C-1001", "item": "September bill", "amount": 74.99, "status": "charged_twice", "delivered_at": "2026-09-01"},
          "ORD-5002": {"order_id": "ORD-5002", "customer_id": "C-1002", "item": "WiFi router", "amount": 320.00, "status": "delivered", "delivered_at": "2026-07-15"},
          "ORD-5003": {"order_id": "ORD-5003", "customer_id": "C-1001", "item": "Mesh extender", "amount": 89.00, "status": "delivered", "delivered_at": "2026-09-20"}}
TICKETS = [{"ticket_id": "T-77", "customer_id": "C-1001", "status": "open", "summary": "Router outage every evening since 20 Sep"},
           {"ticket_id": "T-78", "customer_id": "C-1002", "status": "closed", "summary": "Billing address change"},
           {"ticket_id": "T-81", "customer_id": "C-1002", "status": "open", "summary": "Router reboot loop after firmware update"}]

def fail(category, retryable, message, customer_message=None):
    raise ToolError(json.dumps({"errorCategory": category, "isRetryable": retryable,
                                "message": message, "customerMessage": customer_message}))

@mcp.tool()
def get_customer(query: str) -> str:
    """Find a customer by customer ID (C-1234) or email. Returns a list of matches and a count.
    More than one match means you must ask for another identifier. Do not use for orders or tickets."""
    q = query.strip().lower()
    hits = [c for c in CUSTOMERS.values() if q in (c["customer_id"].lower(), c["email"])]
    return json.dumps({"matches": hits, "count": len(hits)})

@mcp.tool()
def lookup_order(order_id: str) -> str:
    """Look up ONE order by ID (format ORD-1234). Returns item, amount (USD), status, and delivered_at (ISO date).
    Use for order, bill, and return questions. Do not use for account details (use get_customer)."""
    oid = order_id.strip().upper()
    if not oid.startswith("ORD-"):
        fail("validation", False, f"'{order_id}' is not an order ID (expected ORD-1234).", "Could you share the order number?")
    o = ORDERS.get(oid)
    return json.dumps(o if o else {"matches": [], "count": 0, "note": "query succeeded, no such order"})

@mcp.tool()
def process_refund(customer_id: str, order_id: str, amount: float, reason: str) -> str:
    """Submit a refund for a verified customer's order. Refunds over 500 USD are rejected: use escalate_to_human."""
    if amount > 500:
        fail("business", False, "Refunds over 500 USD need a human agent.", "I'll pass this to a specialist.")
    return json.dumps({"refund_id": "RF-0001", "status": "submitted", "amount": amount})

@mcp.tool()
def escalate_to_human(customer_id: str, summary: str, recommended_action: str) -> str:
    """Hand the case to a human agent. The summary must stand alone: the human cannot see this chat."""
    return json.dumps({"handoff_id": "H-0001", "queue": "billing-specialists"})

@mcp.tool()
def server_status() -> str:
    """Report whether the server received its API token (for configuration checks). Never reveals the token."""
    tok = os.environ.get("SUPPORT_API_TOKEN", "")
    return json.dumps({"token_present": bool(tok) and not tok.startswith("${"), "token_length": len(tok)})

@mcp.tool(description=os.environ.get("TICKET_DESC", "Search tickets."))
def search_tickets(text: str, status: str = "any") -> str:
    words = [w for w in text.lower().split() if len(w) > 3]
    hits = [t for t in TICKETS if any(w in t["summary"].lower() for w in words) and status in ("any", t["status"])]
    return json.dumps({"tickets": hits, "count": len(hits)})

@mcp.resource("policies://refunds")
def refund_policy() -> str:
    """Refund policy: windows, thresholds, and exceptions."""
    return ("Refund window: 30 days from delivery. Duplicate charges: always refundable. "
            "Refunds over 500 USD: human approval. Competitor price matching: not covered by policy.")

@mcp.resource("schema://orders")
def order_schema() -> str:
    """Field list for order records returned by lookup_order."""
    return json.dumps({"order_id": "ORD-####", "customer_id": "C-####", "item": "str", "amount": "USD float",
                       "status": "delivered | charged_twice | pending", "delivered_at": "ISO date"})

if __name__ == "__main__":
    mcp.run()

## 2. Test with the MCP client

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
def is_err(res):  # is_error in MCP SDK 2.x, isError in 1.x
    return getattr(res, "is_error", getattr(res, "isError", None))
async def mcp_call(calls, env_extra=None):
    params = StdioServerParameters(command=sys.executable, args=["/content/support_backend.py"], env={**os.environ, **(env_extra or {})})
    out = {}
    async with stdio_client(params) as (r, w):
        async with ClientSession(r, w) as s:
            await s.initialize()
            out["tools"] = [(t.name, t.description) for t in (await s.list_tools()).tools]
            out["resources"] = [str(x.uri) for x in (await s.list_resources()).resources]
            for name, args in calls:
                res = await s.call_tool(name, args); out[(name, json.dumps(args))] = (is_err(res), res.content[0].text)
            out["policy"] = (await s.read_resource("policies://refunds")).contents[0].text
    return out

In [ ]:
CALLS = [("lookup_order", {"order_id": "ORD-5003"}), ("lookup_order", {"order_id": "5003"}), ("lookup_order", {"order_id": "ORD-9999"}),
         ("process_refund", {"customer_id": "C-1002", "order_id": "ORD-5002", "amount": 750, "reason": "duplicate"}), ("server_status", {})]
result = await mcp_call(CALLS, {"SUPPORT_API_TOKEN": "demo-token-123"})
print("TOOLS:"); [print(f"  {n}: {d[:80]}") for n, d in result["tools"]]
print("RESOURCES:", result["resources"])
for (name, args), (err, text) in [(k, v) for k, v in result.items() if isinstance(k, tuple)]:
    print(f"{name}{args} -> is_error={err} | {text[:110]}")

**Check yourself:**
- `ORD-5003` succeeds; `5003` fails with a **validation** error carrying structured metadata.
- `ORD-9999` is a **successful empty result**, not an error.
- A 750 USD refund is rejected by a **business rule**, with a customer-safe message.

## 3. Resources: content by URI

In [ ]:
print(result["policy"])

## 4. Improve a description
`search_tickets` reads its description from an environment variable, so you can compare a weak and a strong version without editing code.

In [ ]:
WEAK = "Search tickets."
STRONG = ("Search LIVE support tickets by keywords. Returns ticket_id, customer_id, status (open or closed), and summary. "
          "Use this instead of searching exported files, which are stale. Pass status='open' for open tickets only.")
for label, desc in (("weak", WEAK), ("strong", STRONG)):
    r = await mcp_call([("search_tickets", {"text": "router problems", "status": "open"})], {"TICKET_DESC": desc})
    print(f"{label}: description seen by hosts -> {dict(r['tools'])['search_tickets'][:90]}")
    print("   result:", list(v for k, v in r.items() if isinstance(k, tuple))[0][1][:120])

Lab 7B shows how the description changes what Claude actually chooses.

## You should now have
- [ ] A working MCP server with tools and resources
- [ ] Client tests for success, validation errors, empty results, and business rules
- [ ] A strong tool description

## Check yourself
1. Which primitive would you use for a list of policies, and why?
2. Why test with the MCP client before connecting a model?
3. What makes an error message useful to Claude?